### New experiments using OSM. 
* We will pull out the relevant data in tags of osm
* Then we will filter to Exeter

In [ ]:
import pyrosm
import geopandas as gpd
import pandas as pd

PBF_FILE = r"osm_data\devon-260326.osm.pbf"

print("Reading OSM data...")
osm = pyrosm.OSM(PBF_FILE)

# Helper function to safely save a layer
def save_layer(gdf, name):
    if gdf is not None and len(gdf) > 0:
        print(f"{name}: {len(gdf):,} features")
        gdf.to_file(f"devon_{name}.gpkg", driver="GPKG")
        print(f"✓ Saved devon_{name}.gpkg")
    else:
        print(f"{name}: empty or not found")

# ── Buildings ─────────────────────────────────────────────
print("\nReading buildings...")
buildings = osm.get_buildings()
save_layer(buildings, "buildings")

# ── Network / Roads ───────────────────────────────────────
print("\nReading roads/network...")
network = osm.get_network(network_type="all")
save_layer(network, "network")

# ── Landuse ───────────────────────────────────────────────
print("\nReading landuse...")
landuse = osm.get_landuse()
save_layer(landuse, "landuse")

# ── POIs ──────────────────────────────────────────────────
print("\nReading POIs...")
pois = osm.get_pois()
save_layer(pois, "pois")

# ── Natural features ──────────────────────────────────────
print("\nReading natural features...")
natural = osm.get_natural()
save_layer(natural, "natural")

# ── Waterways ─────────────────────────────────────────────
print("\nReading waterways...")
waterways = osm.get_data_by_custom_criteria(custom_filter={"waterway": True})
save_layer(waterways, "waterways")

# ── Boundaries ────────────────────────────────────────────
print("\nReading boundaries...")
boundaries = osm.get_data_by_custom_criteria(custom_filter={"boundary": True})
save_layer(boundaries, "boundaries")

print("\nDone!")

### Post filtering of OSM data 
* Removing irrelevant columns
* Finding filter columns

#### Buildings

In [ ]:
import geopandas as gpd
gdf_polygon = gpd.read_file(r"osm_data\devon_buildings.gpkg")
gdf_polygon.columns

In [ ]:
recommended_columns = ['addr:city', 'addr:country', 'addr:housenumber', 'addr:housename',
       'addr:postcode', 'addr:place', 'addr:street','name',
       'opening_hours', 'website', 'building', 'amenity', 'building:flats', 'building:levels',
       'building:material', 'building:min_level', 'building:use','craft',
       'height', 'internet_access', 'landuse', 'levels', 'office', 'shop',
       'source','geometry']

In [ ]:
gdf_polygon.building.isna().any()


In [ ]:
invalid = gdf_polygon[~gdf_polygon.is_valid]
print(len(invalid))

In [ ]:
gdf_polygon.building.unique()

#### Landuse

In [ ]:
gdf_landuse = gpd.read_file(r"osm_data\devon_landuse.gpkg")
gdf_landuse.columns

In [ ]:
invalid = gdf_landuse[~gdf_landuse.is_valid]
print(len(invalid))

In [ ]:
gdf_landuse.crs

In [ ]:
gdf_landuse.head()

In [ ]:
gdf_landuse.landuse.unique()

In [ ]:
gdf_landuse.landuse.isna().any()

#### Natural

In [ ]:
gdf_natural = gpd.read_file(r"osm_data\devon_natural.gpkg")
gdf_natural.columns

In [ ]:
invalid = gdf_natural[~gdf_natural.is_valid]
print(len(invalid))

In [ ]:
gdf_natural.crs

In [ ]:
gdf_natural.natural.unique()

#### POI

In [ ]:
gdf_pois = gpd.read_file(r"osm_data\devon_pois.gpkg")
gdf_pois.columns

In [ ]:
invalid = gdf_pois[~gdf_pois.is_valid]
print(len(invalid))

In [ ]:
gdf_pois.crs

#### waterways

In [ ]:
gdf_waterways = gpd.read_file(r"osm_data\devon_waterways.gpkg")
gdf_waterways.columns

In [ ]:
invalid = gdf_waterways[~gdf_waterways.is_valid]
print(len(invalid))

In [ ]:
gdf_waterways.crs

In [ ]:
gdf_waterways.waterway.unique()

### Boundaries

In [ ]:
gdf_boundaries = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_boundaries.columns

In [ ]:
invalid = gdf_boundaries[~gdf_boundaries.is_valid]
print(len(invalid))

In [ ]:
gdf_boundaries.crs

In [ ]:
gdf_boundaries.name.unique()

### Finally converting all to EPSG 27700

In [ ]:
import os
import geopandas as gpd
for filename in os.listdir(r"osm_data"):
    if filename.endswith(".gpkg"):
        gdf = gpd.read_file(os.path.join(r"osm_data", filename))
        gdf = gdf.to_crs(epsg=27700)
        gdf = gdf[gdf.is_valid]
        gdf.to_file(os.path.join(r"osm_data", filename), driver="GPKG")

In [ ]:
import geopandas as gpd
gdf_admin = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_admin.name.value_counts()

In [ ]:
print(gdf_admin.name.value_counts())

In [ ]:
gdf_admin[gdf_admin.name == "Dorset"]

In [ ]:
import joblib
data = joblib.load(r"artifacts\waterway_river_clyst_exeter_search.pkl").data
data.columns

In [ ]:
data

### DOCUMENTATION : How to use this tool

In [ ]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import joblib
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"Question_0",diff_dir=None).initialize_all_agents()

human_send_message(message="Brilliant, can you also add the polygon of bickington to the map?",target_agent=[agent_archiecture["host_agent"]])

Openai and ngd key set successfully
Model chosen o3
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3


["Your updated map is ready.\n\n• Artifact: bickington_buildings_buffer_map  \n  – An interactive map (HTML) displaying:  \n    1. The Bickington civil-parish boundary polygon (red outline).  \n    2. A 1 km buffer ring around that boundary (light blue shading).  \n    3. All 775 building footprints that fall inside the buffer (blue polygons).\n\nOpen bickington_buildings_buffer_map.html to explore the boundary, buffer, and buildings together.Addtionally some data artifacts have been generated with names  ['bickington_buildings_buffer_map'] and \n descriptions ['Interactive folium map with Bickington boundary, 1km buffer ring, and all buildings within 1km of the boundary.']",

In [ ]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import os
import geopandas as gpd
import numpy as np

gdf = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
available_locations = gdf.name.unique()
available_locations = np.array([loc.lower() for loc in available_locations if loc is not None])
questions = pd.read_csv(r"evaluation\valid.csv")["query"].tolist()
questions = questions[2:]

question_number = 38
for question in questions[38:]:  # Start from the 33rd question (index 32)

    if "OSID" in question:
        continue
    

    location_randomly_sampled = np.random.choice(available_locations)

    question = question.replace("{input_location}",location_randomly_sampled)
    question = question.replace("{{location:Romsey}}",location_randomly_sampled)


    config = None
    with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
        config = json.load(file)

    # Now that things are initialised
    agent_archiecture = OSAgentsInitializer(config,f"Question_{question_number}",diff_dir=None).initialize_all_agents()
    print(f"Starting Question {question_number} : {question}")

    human_send_message(message=question,target_agent=[agent_archiecture["host_agent"]])
    shutil.rmtree("artifacts")
    shutil.rmtree("message_store")
    os.makedirs("artifacts", exist_ok=True)
    os.makedirs("message_store", exist_ok=True)
    print(f"Completed Question {question_number}")
    print("-"*50)
    question_number += 1

#### Local LLMs usage and experiments
* Phase 1 model choices

In [ ]:
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"log",diff_dir=None).initialize_all_agents()

human_send_message(message="Find all buildings in whitchurch that are over 3m tall",target_agent=[agent_archiecture["host_agent"]])
shutil.rmtree("message_store")
shutil.rmtree("artifacts")
os.makedirs("message_store",exist_ok=True)
os.makedirs("artifacts",exist_ok=True)

Openai and ngd key set successfully
Openai and ngd key set successfully
Model chosen o3
Model chosen o3
Model chosen gpt-4o-mini
Model chosen o3
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3


### Questions OS phase 1:

* Where can I buy clothes in Exeter?
* What's the closest food shop to my location?
* Supermarkets near Unite Students accommodation in Exeter?
* Which building is the IAIS?
* How many residential buildings in Exeter ?
* What is north-west of the University of Exeter?
* Where can I park in  central Exeter ?
* Show me all the cairns in Scotland?
* How many primary schools are there on the east side of Bristol?
* How many woodlands are within 10m of the river Dart?
* What are the greenspace in Southampton?
* Show me the hospitals and medical centres in Bristol ?
* List the community centres in the Tyne and Wear ? 
* Which neighbourhoods are near Shirley ?
* Solar farms in London area
*  Shops open Sunday at 15:30 near Exeter Cathedral
* Where can I park my car in Exeter
* Youth hostels in Snowdonia
* Child friendly activities to do in Edinburgh
* Lakes near University of Reading
* Bars located in a nice spot (along the river, nice view,...) in Lake District
* Castles that I can visit in Devon
* Where are the parks near Glasgow in Scotland ?
* Beaches within a mile of a train station in South West UK
* Exeter banks farther than a mile to any police station
* Where can I hire a bicycle in Exeter ?
* Do we have dog friendly café in Exeter ?
* Where are vegan friendly restaurants near University of Exeter ?
* Give me the nearest Health Center near John Lewis in Exeter ?
* Give me locations of 24 hrs gas station in Exeter ?
* Where can I go play tennis in Exeter ?

In [ ]:
questions = """Where can I buy clothes in <location>?
* What's the closest food shop to my <location>?
* Supermarkets near Unite Students accommodation in <location>?
* Which building is the Institute of Arabic and Islamic Studies in <location>?
* How many residential buildings in <location> ?
* Show me all the cairns in <location>?
* How many primary schools are there on the east side of <location>?
* How many woodlands are within 10m of the river Exe?
* What are the greenspace in <location>?
* Show me the hospitals and medical centres in <location> ?
* List the community centres in the Tyne and Wear ? 
* Which neighbourhoods are near Shirley ?
* Solar farms in <location> area
* Where can I park my car in <location>
* Youth hostels in Devon
* Child friendly activities to do in <location> !
* Lakes near University of Exeter
* Bars located in a nice spot (along the river, nice view,...) in <location>
* Castles that I can visit in Devon
* Where are the parks near <location>?
* Beaches within a mile of a train station in South West Devon
* Exeter banks farther than a mile to any police station
* Where can I hire a bicycle in Exeter ?
* Do we have dog friendly café in Exeter ?
* Where are vegan friendly restaurants near University of Exeter ?
* Give me the nearest Health Center near John Lewis in Exeter ?
* Give me locations of 24 hrs gas station in Exeter ?
* Where can I go play tennis in Exeter ?"""

questions = questions.split("*")
questions = [q.replace("\n","").strip() for q in questions if q.strip()]
questions

['Where can I buy clothes in <location>?',
 "What's the closest food shop to my <location>?",
 'Supermarkets near Unite Students accommodation in <location>?',
 'Which building is the Institute of Arabic and Islamic Studies in <location>?',
 'How many residential buildings in <location> ?',
 'Show me all the cairns in <location>?',
 'How many primary schools are there on the east side of <location>?',
 'How many woodlands are within 10m of the river Exe?',
 'What are the greenspace in <location>?',
 'Show me the hospitals and medical centres in <location> ?',
 'List the community centres in the Tyne and Wear ?',
 'Which neighbourhoods are near Shirley ?',
 'Solar farms in <location> area',
 'Where can I park my car in <location>',
 'Youth hostels in Devon',
 'Child friendly activities to do in <location>',
 'Lakes near University of Exeter',
 'Bars located in a nice spot (along the river, nice view,...) in <location>',
 'Castles that I can visit in Devon',
 'Where are the parks near <l

* 11
* 

In [5]:
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import os
import geopandas as gpd
import numpy as np

gdf = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
available_locations = gdf.name.unique()
available_locations = np.array([loc.lower() for loc in available_locations if loc is not None])

question_number = 0
for i, question in enumerate(questions):
    config = None
    
    with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
        config = json.load(file)

    # Now that things are initialised
    agent_archiecture = OSAgentsInitializer(config,f"Question_{question_number}",diff_dir="evaluation/phase_1/phase_1_logs").initialize_all_agents()
    question = question.replace("<location>", np.random.choice(available_locations))
    print(f"Starting Question {question_number} : {question}")

    human_send_message(message=question,target_agent=[agent_archiecture["host_agent"]])
    shutil.rmtree("artifacts")
    shutil.rmtree("message_store")
    os.makedirs("artifacts", exist_ok=True)
    os.makedirs("message_store", exist_ok=True)
    print(f"Completed Question {question_number}")
    question_number += 1
    print("-"*50)
    if question_number == 5:
        break

Openai and ngd key set successfully
Starting Question 0 : Where can I buy clothes in strete?
Model chosen o3
Model chosen o3
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen gpt-4.1


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 9325: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\traitlets\config\application.p

Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4o
Model chosen o3
Completed Question 0
--------------------------------------------------
Starting Question 1 : What's the closest food shop to my nymet rowland?
Model chosen o3
Model chosen gpt-4o-mini
Model chosen o3
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen o3
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen o3
Model chosen gpt-4o
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt

In [2]:
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"Question_16",diff_dir="evaluation/phase_1/phase_1_logs").initialize_all_agents()

human_send_message(message="Lakes near University of Exeter",target_agent=[agent_archiecture["host_agent"]])
shutil.rmtree("message_store")
shutil.rmtree("artifacts")
os.makedirs("message_store",exist_ok=True)
os.makedirs("artifacts",exist_ok=True)

Openai and ngd key set successfully
Model chosen o3
Model chosen o3
Messages for human to process: For your query "Lakes near University of Exeter" could you please specify what distance you would consider as "near"? For example, within 1 km, 3 km, 5 km, etc. This will help me provide results that match your expectations.
OFFLINE MODE : Please provide your response to the following query:
Model chosen o3
Model chosen gpt-4.1
bbox is  None <class 'NoneType'> True <class 'bool'>
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen o3
Model chosen gpt-4o
Model chosen gpt-4o


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 14991: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\traitlets\config\application.

Model chosen gpt-4o
Model chosen gpt-4.1


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 14991: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\traitlets\config\application.

Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4o
Model chosen o3
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen o3


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\ab1574\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2011' in position 40619: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\ab1574\OneDrive - University of Exeter\Desktop\Ordnance_Survey\osvenv\Lib\site-packages\traitlets\config\application.

Model chosen o3
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4o
Model chosen gpt-4o
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3


In [2]:
import geopandas as gpd
gdf = gpd.read_file(r"osm_data\devon_pois.gpkg")
gdf.columns

Index(['visible', 'tags', 'lon', 'timestamp', 'lat', 'version', 'id',
       'changeset', 'addr:city', 'addr:country', 'addr:full',
       'addr:housenumber', 'addr:housename', 'addr:postcode', 'addr:place',
       'addr:street', 'email', 'name', 'opening_hours', 'operator', 'phone',
       'ref', 'url', 'website', 'amenity', 'atm', 'bank', 'bicycle_parking',
       'bicycle_rental', 'bar', 'building', 'building:levels', 'cafe',
       'college', 'drinking_water', 'fast_food', 'fountain', 'fuel',
       'gambling', 'internet_access', 'landuse', 'office', 'parking',
       'post_office', 'school', 'social_facility', 'source', 'start_date',
       'wikipedia', 'alcohol', 'appliance', 'bicycle', 'charity', 'clothes',
       'confectionery', 'craft', 'farm', 'fireplace', 'furniture',
       'garden_centre', 'gift', 'hairdresser', 'motorcycle', 'music',
       'organic', 'outdoor', 'religion', 'second_hand', 'shoes', 'shop',
       'tattoo', 'trade', 'wholesale', 'attraction', 'camp_site',


In [4]:
gdf[gdf.name.str.contains("nhs",case=False,na=False)].name

4416                                    NHS Walk in Centre
4922                            Addy Dental Practise (NHS)
28568    Torbay & South Devon NHS Foundation Trust Parking
41137                             NHS Nightingale Hospital
Name: name, dtype: object

In [5]:
gdf[gdf.name.str.contains("nhs",case=False,na=False)].amenity

4416       clinic
4922      dentist
28568     parking
41137    hospital
Name: amenity, dtype: object